<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "690"
iso_country = "Seychelles"
year = 2022
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 2.000*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Seychelles': 

Number of trade records present in dataset: 23872
Number of partner countries present in dataset: 126
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1932.5m
Total weight within the dataset: 1022.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,690,Seychelles,699,India,Import,Sea,551349,"551349 | Fabrics, woven; prin",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,17827.038,NaN,17827.038,kg,792.03,792.03,2747.146556,3541.0,2200.274711,6044.477094,['Container'],['Container'],22.871894,22.821993,22.827379,22.827379,22.849514,22.849514,23.998115,43.418360,25.882516,24.655432,24.733932,25.233825,23.466478,42.195495,27.603245,25.493577,24.838292,26.063965,23.000082,46.938346,34.459432,30.239893,27.864582,27.211419,22.542423,49.541881,40.659386,37.378613,29.857728,29.226755,22.093487,51.171331,49.610004,50.513811,31.665259,30.729003,21.635574,49.955787,46.865852,50.694276,39.834381,31.316556,21.199855,43.685905,46.106256,48.315226,34.457514,31.688508,20.760080,40.125117,40.569095,46.440825,39.143129,31.925437,2676
1,1,690,Seychelles,480,Mauritius,Import,Sea,551349,"551349 | Fabrics, woven; prin",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,2614.081,NaN,2614.081,kg,300.00,300.00,402.829882,1909.0,1186.197239,477.835694,['Container'],['Container'],1.808098,1.804153,1.804579,1.804579,1.806329,1.806329,1.897129,3.432363,2.046098,1.949093,1.955298,1.994816,1.855102,3.335692,2.182127,2.015351,1.963548,2.060442,1.818232,3.710630,2.724131,2.390563,2.202786,2.151152,1.782052,3.916448,3.214257,2.954902,2.360351,2.310471,1.746562,4.045261,3.921833,3.993282,2.503242,2.429228,1.710363,3.949168,3.704899,4.007548,3.149038,2.475676,1.675918,3.453514,3.644850,3.819477,2.723979,2.505080,1.641152,3.172022,3.207120,3.671299,3.094392,2.523810,2676


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $287.9m (14.9%), ... 

                              cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
03 | Fish and crustaceans  143710777.0  67608010.3      7.4      6.6
02 | Meat and edible meat   19946266.8   7138505.2      1.0      0.7
04 | Dairy produce; birds   19027815.9   9256886.9      1.0      0.9
15 | Animal, vegetable or   16763007.9   9780931.0      0.9      1.0
20 | Preparations of vege   11332841.2   7874039.0      0.6      0.8
19 | Preparations of cere   10767660.7   4221026.2      0.6      0.4
10 | Cereals                10178075.7  23613205.9      0.5      2.3
22 | Beverages, spirits a    8644578.1  31448597.6      0.4      3.1
21 | Miscellaneous edible    8382336.5   2684973.4      0.4      0.3
23 | Food industries, res    7217559.8   8299380.1      0.4      0.8
07 | Vegetables and certa    6114263.0  10668172.3      0.3      1.0
11

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $281.3m (14.6%), ... 

                     cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Fishing           139983511.4  67048607.4      7.2      6.6
Agriculture        82362881.6  74069209.2      4.3      7.2
Food & Beverages   58923718.4  61419874.4      3.0      6.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $272.9m (14.1%), ... 

                                  cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
OFD - Food products n.e.c.     186734372.5  99979186.6      9.7      9.8
MIL - Dairy products            18745878.8   8172811.8      1.0      0.8
VOL - Vegetable oils and fats   17965002.2  12675370.5      0.9      1.2
OMT - Meat products n.e.c.      15520076.3   6227884.8      0.8      0.6
CMT - Bovine meat prods          8597294.7   1702085.8      0.4      0.2
V_F - Vegetables, fruit, nuts    8379191.1  12433859.9      0.4      1.2
PCR - Processed rice             7554999.1  13123033.4      0.4      1.3
OCR - Crops n.e.c.               4190041.1   1858336.9      0.2      0.2
SGR - Sugar                      2447835.5   5358347.8      0.1      0.5
GRO - Cereal grains n.e.c.       2142486.6   8845898.8      0.1      0.9
OAP - Animal products n.e.c.      482008.3    174282.5    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 14.9% 
GTAP 14.1% 
EORA 14.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46416/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,690,Seychelles,699,India,Import,Sea,551349,"551349 | Fabrics, woven; prin",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,17827.038,NaN,17827.038,kg,792.03,792.03,2747.146556,3541.0,2200.274711,6044.477094,['Container'],['Container'],22.871894,22.821993,22.827379,22.827379,22.849514,22.849514,23.998115,43.418360,25.882516,24.655432,24.733932,25.233825,23.466478,42.195495,27.603245,25.493577,24.838292,26.063965,23.000082,46.938346,34.459432,30.239893,27.864582,27.211419,22.542423,49.541881,40.659386,37.378613,29.857728,29.226755,22.093487,51.171331,49.610004,50.513811,31.665259,30.729003,21.635574,49.955787,46.865852,50.694276,39.834381,31.316556,21.199855,43.685905,46.106256,48.315226,34.457514,31.688508,20.760080,40.125117,40.569095,46.440825,39.143129,31.925437,2676,NaN
1,1,690,Seychelles,480,Mauritius,Import,Sea,551349,"551349 | Fabrics, woven; prin",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,2614.081,NaN,2614.081,kg,300.00,300.00,402.829882,1909.0,1186.197239,477.835694,['Container'],['Container'],1.808098,1.804153,1.804579,1.804579,1.806329,1.806329,1.897129,3.432363,2.046098,1.949093,1.955298,1.994816,1.855102,3.335692,2.182127,2.015351,1.963548,2.060442,1.818232,3.710630,2.724131,2.390563,2.202786,2.151152,1.782052,3.916448,3.214257,2.954902,2.360351,2.310471,1.746562,4.045261,3.921833,3.993282,2.503242,2.429228,1.710363,3.949168,3.704899,4.007548,3.149038,2.475676,1.675918,3.453514,3.644850,3.819477,2.723979,2.505080,1.641152,3.172022,3.207120,3.671299,3.094392,2.523810,2676,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $102.1m of $1932.5m (5.3%)
Volume: 91.5 of 1022.0 million kilograms (9.0%)
Value-to-Weight: 1.12 from 1.89 US$ per kg (+-41.0%)
Average distance to market (volume-weighted): 3686.6 from 3169.6 nm (+16.3%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    899  32877583.6  32384535.5      1.7      3.2
1                Dairy    218  18597591.3   9125420.4      1.0      0.9
2        Fats and Oils    175  16634566.9   9634931.3      0.9      0.9
3        Meat and Prep    128  13550071.2   5938437.9      0.7      0.6
4           Vegetables    319   9767892.7  13713392.3      0.5      1.3
5      Sugar and Honey    175   4187692.3   7144940.1      0.2      0.7
6               Fruits    222   3879446.0   3138710.0      0.2      0.3
7  Unprocessed Cereals     66   2595005.2  10441412.3      0.1      1.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,17556787.6,31563283.9,18919020.5,18023228.9,18077695.0,18448563.7,16182883.2,37152459.7,35950552.4,36623220.4,23108849.0,22418680.8,15221147.0,29330948.0,29639655.5,33859757.7,28449199.9,23286319.6
1,Fats and Oils,17860514.3,32146440.3,19249124.3,18338487.1,18394287.3,18770141.2,16459167.7,37850720.1,36632119.3,37313306.5,23518945.7,22818884.4,15478200.8,29851982.1,30168453.5,34471761.3,28972755.8,23703383.8
2,Fruits,4792690.6,8612646.2,5164258.6,4920512.7,4935183.4,5036083.2,4417898.3,10137628.0,9803246.2,9985394.3,6306810.4,6119765.0,4155546.7,8015304.1,8098485.8,9245173.7,7758593.4,6356977.2
3,Meat and Prep,16607207.2,30046431.4,17911253.1,17062084.8,17116408.5,17462344.8,15289164.0,35411652.3,34331181.2,34956634.9,21913034.4,21265125.6,14366418.1,27767436.8,28074678.8,32138041.7,27087879.9,22093083.5
4,Processed Cereals,38434453.2,69209943.8,41420028.8,39458534.7,39581266.5,40392264.3,35416146.6,81568380.8,78996173.4,80467089.8,50655820.4,49103785.9,33303219.2,64163361.5,64856332.4,74172654.7,62446868.9,51001739.9
5,Sugar and Honey,10273384.9,18332115.3,11052120.5,10528668.5,10561120.3,10784921.9,9483284.1,21642653.3,20940138.1,21349196.8,13528959.6,13062853.0,8930226.1,17055770.6,17236860.9,19705724.9,16604309.8,13557270.0
6,Unprocessed Cereals,10391665.1,18339644.9,11155020.4,10626818.2,10659441.3,10894588.5,9612820.5,21705135.6,20981442.4,21415087.0,13676182.8,13135818.6,9067721.9,17129215.9,17308192.4,19784510.4,16697070.6,13619422.0
7,Vegetables,20116858.3,36001613.5,21663440.5,20634782.2,20695527.2,21127299.8,18559866.4,42365731.6,40926129.2,41708107.4,26442839.9,25634681.0,17467422.7,33562398.7,33917962.7,38714224.0,32456548.9,26620489.9


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.75,0.07,0.03,0.03,0.05,1.13,1.06,1.10,0.37,0.34,0.76,0.78,1.00,0.71,0.43
1,Fats and Oils,0.86,0.08,0.03,0.03,0.05,1.29,1.21,1.25,0.42,0.38,0.86,0.88,1.14,0.81,0.49
2,Fruits,0.98,0.10,0.03,0.04,0.06,1.47,1.39,1.44,0.49,0.44,0.99,1.02,1.31,0.93,0.57
3,Meat and Prep,0.99,0.10,0.03,0.04,0.06,1.49,1.41,1.45,0.49,0.44,0.99,1.01,1.31,0.94,0.57
4,Processed Cereals,0.94,0.09,0.03,0.03,0.06,1.40,1.33,1.37,0.46,0.42,0.94,0.96,1.24,0.89,0.54
5,Sugar and Honey,1.92,0.19,0.06,0.07,0.12,2.90,2.74,2.83,0.97,0.85,1.94,1.98,2.57,1.83,1.10
6,Unprocessed Cereals,3.06,0.29,0.09,0.10,0.19,4.66,4.38,4.55,1.57,1.36,3.11,3.18,4.13,2.94,1.75
7,Vegetables,1.63,0.16,0.05,0.06,0.10,2.44,2.29,2.37,0.81,0.72,1.65,1.68,2.18,1.53,0.94
